# M01. Weather Factors

### Imports

In [1]:
from DataImports import *

### Settings

In [2]:
ensemble_size = 10              # Number of classifiers in the model ensemble
save_event_averages = False     # Save event averages for use in WFX calculations
num_similar_weather_games = 50  # Number of similar weather games to use for WFX calibrations

### Data

##### Plate Appearances

In [3]:
pa_dataset = pd.read_csv(os.path.join(baseball_path, "PA Dataset.csv"))

##### Open Meteo

In [4]:
open_meteo_column_list = ['game_id', 'year', 'venue_name', 'location.defaultCoordinates.latitude', 'location.defaultCoordinates.longitude', 
                          'fieldInfo.leftLine', 'fieldInfo.center', 'fieldInfo.rightLine', 'fieldInfo.leftCenter', 'fieldInfo.rightCenter', 
                          'location.elevation', 'location.azimuthAngle', 'fieldInfo.roofType', 'active', 
                          'temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'surface_pressure', 
                          'wind_speed_10m', 'wind_direction_10m', 'weather_code', 'precipitation_probability']

weather_df = pd.concat(map(pd.read_csv, glob.glob(os.path.join(baseball_path, "A06. Weather", "1. Open Meteo", "*.csv"))), ignore_index=True)[open_meteo_column_list]

##### PFX

In [5]:
shifted_game_pfx_df = pd.read_csv(os.path.join(baseball_path, "PFX.csv"))

### Merge

##### MLB Stats API with Open Meteo

In [6]:
weather_dataset = pa_dataset.merge(weather_df.drop(columns=['year']), left_on=['gamePk'], right_on=['game_id'], how='inner')

##### With PFX

In [7]:
pfx_list = [f"{event}_pfx" for event in events_list]
weather_dataset = weather_dataset.merge(shifted_game_pfx_df[['gamePk', 'batSide'] + pfx_list], on=['gamePk', 'batSide'], how='left')

### Clean

##### Open Meteo

Calculate wind vectors

In [8]:
def calculate_vectors(row, azimuth_column, wind_column, speed_column):
    angle = row[wind_column] - row[azimuth_column]
    
    # Calculate vectors
    x_vect = round(math.sin(math.radians(angle)), 5) * row[speed_column] * -1
    y_vect = round(math.cos(math.radians(angle)), 5) * row[speed_column] * -1

    return pd.Series([x_vect, y_vect], index=['x_vect', 'y_vect'])

In [9]:
weather_dataset[['meteo_x_vect', 'meteo_y_vect']] = weather_dataset.apply(lambda row: calculate_vectors(row, 'location.azimuthAngle', 'wind_direction_10m', 'wind_speed_10m'), axis=1)

Use weather column from MLB data to adjust Open Meteo data for domes/roofs <br>
Note: Second clean stage is required because it requires data from two sources

In [10]:
mask = weather_dataset['weather'].str.contains('Roof|Dome', case=False, na=False)

In [11]:
weather_dataset.loc[mask, 'temperature'] = 70
weather_dataset.loc[mask, 'x_vect'] = 0
weather_dataset.loc[mask, 'y_vect'] = 0

In [12]:
weather_dataset.loc[mask, 'temperature_2m'] = 70
weather_dataset.loc[mask, 'meteo_x_vect'] = 0
weather_dataset.loc[mask, 'meteo_y_vect'] = 0
weather_dataset.loc[mask, 'relative_humidity_2m'] = 60
weather_dataset.loc[mask, 'dew_point_2m'] = 57

### Model

$ \hat{\text{eventsModel2}} = \hat{\text{eventsModel}} + pfx + meteo\_x\_vect + meteo\_y\_vect + temperature\_2m + relative\_humidity\_2m + dew\_point\_2m + surface\_pressure + venue\_id + b\_L$

The purpose of this model is to estimate rates of events in games based on weather and venue. This model is trained with expected rates based on the actual batted ball data. This allows for us to control for differences in inherent batted ball data across games. The model then predicts with league average rates to determine how a game with typical batted ball data would differ in various weather and venue conditions. <br>
Ideally, we would then compare these predicted rates to league average rates to determine park x weather factors, multipliers that estimate how much more or less likely given events are on the game-level than under average conditions. <br>
However, this is hard. <br>
We'll recalibrate rates later using similar weather games

##### Inputs

Meteo weather inputs

In [13]:
meteo_weather_list = ['meteo_x_vect', 'meteo_y_vect', 'temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'surface_pressure']

Parks with sufficient samples

In [14]:
venue_dummy_list = [f'venue_{id}' for id in sorted(weather_dataset['venue_id'].value_counts()[lambda x: x > 20000].index.tolist())]

Predicted rates

In [15]:
events_list_pred_batted = [f"{event}_pred_batted" for event in events_list]

Select inputs

In [16]:
# wfx_inputs = pfx_list + meteo_weather_list + venue_dummy_list + ['b_L']

##### Sample

Create venue dummies

Note: not all venue dummies may be included in venue_dummy_list

In [17]:
weather_dataset['venue_id2'] = weather_dataset['venue_id'].copy()

In [18]:
weather_dataset = pd.get_dummies(weather_dataset, columns=['venue_id2'], prefix='venue', drop_first=False)

Set pfx to 1 if not in venue sample

Note: we may want to set this in shifted_game_pfx_df and default to a rolling value

In [19]:
weather_dataset['sample_venue'] = weather_dataset[venue_dummy_list].sum(axis=1)

In [20]:
for pfx in pfx_list:
    weather_dataset[pfx] = np.where(weather_dataset['sample_venue'] == 0, 1, weather_dataset[pfx])

Group by game

In [21]:
weather_dataset = weather_dataset.groupby(['gamePk', 'date', 'venue_id', 'batSide'])[events_list_pred_batted + pfx_list + meteo_weather_list + venue_dummy_list + ['b_L'] + events_list].mean().reset_index()

In [22]:
eps = 1e-4

for event in events_list:
    weather_dataset[f'{event}_log_multiplier'] = np.log(weather_dataset[event] + eps) - np.log(weather_dataset[f'{event}_pred_batted'] + eps)
    weather_dataset[f'{event}_log_pfx'] = np.log(weather_dataset[f'{event}_pfx'] + eps)

In [23]:
log_multiplier_cols = [col for col in weather_dataset.columns if "log_multiplier" in col]
log_pfx_cols = [col for col in weather_dataset.columns if "log_pfx" in col]

In [24]:
wfx_inputs = log_pfx_cols + meteo_weather_list + venue_dummy_list + ['b_L']

Drop if missing inputs

In [25]:
weather_dataset.dropna(subset=wfx_inputs, inplace=True)

Define lefty batter dummy

In [26]:
weather_dataset['b_L'] = (weather_dataset['batSide'] == "L").astype(int)

Restrict by date

In [27]:
weather_dataset = weather_dataset[(weather_dataset['date'] > 20180101) & (weather_dataset['date'] < 20250101)]

Split features and target

In [28]:
X = weather_dataset[wfx_inputs].values
y = weather_dataset[log_multiplier_cols].values

##### Scale

In [29]:
# Columns to scale
scale_cols = [
    'meteo_x_vect','meteo_y_vect','temperature_2m','relative_humidity_2m',
    'dew_point_2m','surface_pressure'
]

# Get integer column positions
scale_idx = [wfx_inputs.index(c) for c in scale_cols]

if not hasattr(sys.modules['__main__'], '__file__'):
    # Copy X to avoid modifying original
    X_scaled = X.copy()
    
    # Initialize scaler
    scale_wfx = StandardScaler()
    
    # Fit and transform only the selected columns
    X_scaled[:, scale_idx] = scale_wfx.fit_transform(X_scaled[:, scale_idx])
    
    # Create directory
    os.makedirs(os.path.join(model_path, "M01. Park and Weather Factors", todaysdate), exist_ok=True)
    
    # Save scaler
    pickle.dump(scale_wfx, open(os.path.join(model_path, "M01. Park and Weather Factors", todaysdate, "scale_wfx.pkl"), 'wb'))
else:
    # Transform new data with existing scaler
    X_scaled = X.copy()
    X_scaled[:, scale_idx] = scale_wfx.transform(X_scaled[:, scale_idx])

##### Train

In [30]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


def train_one_model(i, X_scaled, y, model_path, todaysdate):
    # Build NN
    model = Sequential([
        Dense(256, input_shape=(X_scaled.shape[1],), activation='relu'),
        Dropout(0.15),
        Dense(128, activation='relu'),
        Dropout(0.15),
        Dense(64, activation='relu'),
        Dense(y.shape[1], activation='linear')  # linear output for log multipliers
    ])

    # Compile with MSE loss for log multiplier regression
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mse']
    )

    # Early stopping
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )

    # Train
    model.fit(
        X_scaled, y,
        epochs=100,
        batch_size=32,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=1
    )

    # Save model
    model_folder = os.path.join(model_path, "M01. Park and Weather Factors", todaysdate)
    os.makedirs(model_folder, exist_ok=True)
    model_path_i = os.path.join(model_folder, f'predict_wfx_{i}.keras')
    model.save(model_path_i)
    
    return model

# Train ensemble in parallel
ensemble_models = Parallel(n_jobs=-1, backend='loky')(
    delayed(train_one_model)(i, X_scaled, y, model_path, todaysdate)
    for i in range(ensemble_size)
)

# Wrap ensemble
class VotingEnsemble:
    def __init__(self, models):
        self.models = models

    def predict(self, X):
        predictions = np.array([model.predict(X, verbose=0) for model in self.models])
        return np.mean(predictions, axis=0)

predict_wfx = VotingEnsemble(ensemble_models)

##### Predict

In [31]:
# Predict log multipliers using the ensemble
log_multiplier_preds = predict_wfx.predict(X_scaled)

# Optional: check shape
print("Predicted log multipliers shape:", log_multiplier_preds.shape)

# Make a copy of the dataframe for WFX predictions
weather_dataset2 = weather_dataset.copy()


for i, event in enumerate(events_list):
    # Uncentered log multiplier
    weather_dataset2[f'{event}_log_multiplier_pred'] = log_multiplier_preds[:, i]
    
    # Uncentered WFX
    weather_dataset2[f'{event}_wfx_unadj'] = np.exp(log_multiplier_preds[:, i])
    
    # Centered log multiplier
    mean_log = weather_dataset2[f'{event}_log_multiplier_pred'].mean()
    weather_dataset2[f'{event}_log_multiplier_pred_centered'] = (weather_dataset2[f'{event}_log_multiplier_pred'] - weather_dataset2.groupby('batSide')[f'{event}_log_multiplier_pred'].transform('mean'))

    # Centered WFX
    weather_dataset2[f'{event}_wfx_adj'] = np.exp(weather_dataset2[f'{event}_log_multiplier_pred_centered'])

Predicted log multipliers shape: (30568, 11)


### Evaluations

Summary Statistics

In [34]:
wfx_cols  = [col for col in weather_dataset2.columns if "_wfx_" in col]
weather_dataset2[wfx_cols].describe()

,b1_wfx_unadj,b1_wfx_adj,b2_wfx_unadj,b2_wfx_adj,b3_wfx_unadj,b3_wfx_adj,bb_wfx_unadj,bb_wfx_adj,fo_wfx_unadj,fo_wfx_adj,go_wfx_unadj,go_wfx_adj,hbp_wfx_unadj,hbp_wfx_adj,hr_wfx_unadj,hr_wfx_adj,lo_wfx_unadj,lo_wfx_adj,po_wfx_unadj,po_wfx_adj,so_wfx_unadj,so_wfx_adj
count,30568.000000,30568.000000,30568.000000,30568.000000,30568.000000,30568.000000,30568.000000,30568.000000,30568.000000,30568.000000,30568.000000,30568.000000,30568.000000,30568.000000,30568.000000,30568.000000,30568.000000,30568.000000,30568.000000,30568.000000,30568.000000,30568.000000
mean,0.869513,1.001132,0.332618,1.024369,0.058167,1.032456,0.999605,1.000001,0.875778,1.001185,0.984330,1.000066,0.998725,1.000002,0.387275,1.075176,0.423746,1.009600,0.568018,1.013706,1.000478,1.000001
std,0.083110,0.047663,0.115436,0.238179,0.018137,0.289847,0.001547,0.001510,0.065582,0.048667,0.013950,0.011489,0.002099,0.002097,0.164496,0.412736,0.119977,0.140153,0.122578,0.161046,0.001437,0.001422
min,0.657732,0.827870,0.112038,0.485500,0.026850,0.507004,0.989800,0.990525,0.675965,0.820462,0.836794,0.843140,0.971493,0.972605,0.047881,0.138982,0.171995,0.553085,0.193590,0.407903,0.993838,0.993162
25%,0.795879,0.975269,0.220118,0.870421,0.046249,0.849995,0.998667,0.999125,0.822310,0.968498,0.974877,0.992746,0.997373,0.998646,0.262848,0.793685,0.309327,0.917994,0.477331,0.916328,0.999595,0.999168
50%,0.898197,0.999245,0.338505,0.981995,0.055885,0.959483,0.999618,0.999990,0.881207,1.001480,0.984996,1.000649,0.998738,0.999992,0.361907,1.013270,0.429402,1.003886,0.569281,1.020084,1.000487,1.000019
75%,0.941052,1.026483,0.426958,1.127886,0.065036,1.120652,1.000454,1.000840,0.930409,1.035025,0.994796,1.007736,1.000049,1.001313,0.478088,1.287061,0.534019,1.089524,0.660635,1.121722,1.001366,1.000868
max,1.189847,1.388819,0.690092,2.444599,0.523195,8.160039,1.031089,1.031843,1.039937,1.249591,1.035812,1.061222,1.013864,1.015296,1.409346,3.743028,0.798407,2.206536,1.099787,1.793606,1.029950,1.029669


Park Outliers

In [36]:
# Sort and get top/bottom 500
top_500 = weather_dataset2.nlargest(500, 'hr_wfx_unadj')
bottom_500 = weather_dataset2.nsmallest(500, 'hr_wfx_unadj')

# Get value counts
top_counts = top_500['venue_id'].value_counts().head(5)
bottom_counts = bottom_500['venue_id'].value_counts().head(5)

# Combine into a 5x4 DataFrame
result_df = pd.DataFrame({
    'Top Venue': top_counts.index,
    'Top Count': top_counts.values,
    'Bottom Venue': bottom_counts.index,
    'Bottom Count': bottom_counts.values
})

result_df

,Top Venue,Top Count,Bottom Venue,Bottom Count
0,2602,173,17,119
1,17,118,2394,80
2,3313,83,3,72
3,2,69,7,57
4,2681,23,2889,43


Best Home Run Games

In [37]:
weather_dataset2.sort_values('hr_wfx_unadj', ascending=False).head(100)[['meteo_y_vect', 'temperature_2m']].mean()

meteo_y_vect     11.239851
temperature_2m   84.859628
dtype: float64

Worst Home Run Games

In [38]:
weather_dataset2.sort_values('hr_wfx_unadj', ascending=False).tail(100)[['meteo_y_vect', 'temperature_2m']].mean()

meteo_y_vect     -9.461999
temperature_2m   51.241158
dtype: float64

##### WFX Dataframe

Convert from long to wide

In [42]:
l_shifted_game_wfx_df = weather_dataset2[weather_dataset2['batSide'] == "L"]
r_shifted_game_wfx_df = weather_dataset2[weather_dataset2['batSide'] == "R"]

wfx_df = pd.merge(l_shifted_game_wfx_df, r_shifted_game_wfx_df, on=['venue_id', 'gamePk', 'date'], how='left', suffixes=("_l", "_r"))

Add venue name

In [46]:
wfx_df = wfx_df.merge(venue_map_df[['id', 'name']], left_on='venue_id', right_on='id', how='left')

Venue Rankings

In [55]:
wfx_df[wfx_df['date'].astype(str) >= '20220101'].groupby(['name', 'venue_id'])[['hr_wfx_unadj_l', 'hr_wfx_unadj_r', 'hr_wfx_adj_l', 'hr_wfx_adj_r']].mean().sort_values('hr_wfx_adj_r', ascending=False)

,,hr_wfx_unadj_l,hr_wfx_unadj_r,hr_wfx_adj_l,hr_wfx_adj_r
name,venue_id,,,,
Great American Ball Park,2602,0.520273,0.683804,1.781898,1.593384
Yankee Stadium,3313,0.353009,0.570148,1.209030,1.328545
Rogers Centre,14,0.270356,0.567166,0.925949,1.321597
Citizens Bank Park,2681,0.380789,0.493838,1.304175,1.150729
Daikin Park,2392,0.249543,0.493292,0.854668,1.149456
American Family Field,32,0.359882,0.490226,1.232569,1.142313
Dodger Stadium,22,0.281440,0.471029,0.963913,1.097580
Coors Field,19,0.291201,0.459642,0.997344,1.071047
Rate Field,4,0.260994,0.449907,0.893887,1.048362


### Write

All Games

In [44]:
wfx_df[['venue_id', 'gamePk', 'date'] + [col for col in wfx_df if "wfx" in col] + [col for col in wfx_df if "pred" in col] + [f'{event}_l' for event in events_list] + [f'{event}_r' for event in events_list]].to_csv(
    os.path.join(baseball_path, "Park and Weather Factors.csv"), index=False)

Individual Games

In [45]:
for date in wfx_df['date'].unique():
    wfx_df[wfx_df['date'] == date][['venue_id', 'gamePk', 'date'] + [col for col in wfx_df if "wfx" in col]].to_csv(os.path.join(baseball_path, "B02. WFX", f"Park and Weather Factors {date}.csv"), index=False)